In [2]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

print("Reloaded:", X_train.shape, X_test.shape)

Reloaded: (5634, 33) (1409, 33)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

              precision    recall  f1-score   support

    No Churn       0.97      0.93      0.95      1035
       Churn       0.83      0.93      0.88       374

    accuracy                           0.93      1409
   macro avg       0.90      0.93      0.91      1409
weighted avg       0.93      0.93      0.93      1409

ROC-AUC: 0.9810380014983595


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=["No Churn", "Churn"]))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_rf))

              precision    recall  f1-score   support

    No Churn       0.96      0.96      0.96      1035
       Churn       0.90      0.90      0.90       374

    accuracy                           0.95      1409
   macro avg       0.93      0.93      0.93      1409
weighted avg       0.95      0.95      0.95      1409

ROC-AUC: 0.9808636234467437


In [5]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss")
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_pred_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb, target_names=["No Churn", "Churn"]))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_xgb))

              precision    recall  f1-score   support

    No Churn       0.96      0.96      0.96      1035
       Churn       0.90      0.90      0.90       374

    accuracy                           0.95      1409
   macro avg       0.93      0.93      0.93      1409
weighted avg       0.95      0.95      0.95      1409

ROC-AUC: 0.9827352812007543


In [6]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(xgb, "../models/xgboost_model.pkl")
print("XGBoost model saved")

XGBoost model saved


In [7]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

xgb_base = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss")

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    scoring="recall",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV recall score:", random_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
Best CV recall score: 0.9210702341137124


In [8]:
best_xgb = random_search.best_estimator_

y_pred_tuned = best_xgb.predict(X_test)
y_pred_proba_tuned = best_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_tuned, target_names=["No Churn", "Churn"]))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_tuned))

              precision    recall  f1-score   support

    No Churn       0.97      0.95      0.96      1035
       Churn       0.87      0.91      0.89       374

    accuracy                           0.94      1409
   macro avg       0.92      0.93      0.93      1409
weighted avg       0.94      0.94      0.94      1409

ROC-AUC: 0.9838668010023509


In [9]:
joblib.dump(best_xgb, "../models/xgboost_tuned_final.pkl")
print("Final tuned model saved")

Final tuned model saved


In [10]:
print(X_train.columns.tolist())

['SeniorCitizen', 'customer_tenure_months', 'monthly_spend', 'support_tickets_last_90d', 'feature_usage_score', 'days_since_last_login', 'subscription_plan_encoded', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'is_high_value_customer', 'engagement_trend']


In [11]:
import pandas as pd
X_train = pd.read_csv("../data/X_train.csv")
print(X_train.columns.tolist())

['SeniorCitizen', 'customer_tenure_months', 'monthly_spend', 'support_tickets_last_90d', 'feature_usage_score', 'days_since_last_login', 'subscription_plan_encoded', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'is_high_value_customer', 'engagement_trend']
